**PHASE 4 - Benchmark Baseline**

**SCOPE**
1. Simple baselines: majority, random stratified, length heuristic, keyword heuristic.
2. Traditional ML: TF-IDF + Logistic Regression / Linear SVM / Naive Bayes / Random Forest.
3. Evaluation trên random split, 5-fold, source-held-out và category-held-out.
4. Export kết quả, predictions, confusion matrices và feature importance.

In [1]:
from __future__ import annotations

import json
import random
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [4]:
def read_required_csv(path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy file bắt buộc: {path}')
    return pd.read_csv(path)

features_df = read_required_csv("/content/viclickbait_eda_features.csv")
random_split_df = read_required_csv("/content/random_stratified_70_10_20.csv")
kfold_df = read_required_csv("/content/stratified_kfold_5.csv")
source_heldout_df = read_required_csv("/content/leave_one_source_out.csv")
category_heldout_df = read_required_csv("/content/category_heldout.csv")

with open("/content/metrics_config.json", encoding='utf-8') as f:
    metrics_config = json.load(f)

print('Feature data:', features_df.shape)
print('Random split:', random_split_df.shape)
print('K-fold split:', kfold_df.shape)
print('Source-held-out:', source_heldout_df.shape)
print('Category-held-out:', category_heldout_df.shape)
print('Primary metric:', metrics_config.get('primary_metric'))

display(features_df.head(3))

Feature data: (3414, 38)
Random split: (3414, 8)
K-fold split: (3414, 7)
Source-held-out: (27312, 8)
Category-held-out: (40968, 8)
Primary metric: macro_f1


,id,title,lead_paragraph,source,category,publish_datetime_raw,publish_dt,label,label_str,title_char_len,...,title_has_quote,title_has_source_tag,lead_char_len,lead_word_len,lead_title_ratio,lead_n_cb_keywords,source_encoded,category_encoded,source_cb_rate_diagnostic,category_cb_rate_diagnostic
0,article_0001,Sân bay Vinh đóng cửa 6 tháng để nâng cấp,Nghệ AnCảng hàng không Vinh sẽ ngừng hoạt động...,VnExpress,Tin tức tổng hợp,2025-06-23T12:26:00+07:00,2025-06-23 05:26:00.000,0,Non-clickbait,41,...,0,0,155,34,3.690476,0,7,10,0.144737,0.040609
1,article_0002,5 người thoát nạn khi ôtô bị lũ cuốn,Tuyên QuangĐi theo Google Maps song lại được c...,VnExpress,Tin tức tổng hợp,2025-06-23T12:26:00+07:00,2025-06-23 05:26:00.000,0,Non-clickbait,36,...,0,0,142,31,3.837838,0,7,10,0.144737,0.040609
2,article_0004,Quy hoạch 9 khu chức năng phía đông TP HCM,9 đồ án quy hoạch 1/2000 được TP Thủ Đức phê d...,VnExpress,Tin tức tổng hợp,2025-06-23T11:56:00+07:00,2025-06-23 04:56:00.000,0,Non-clickbait,42,...,0,0,156,34,3.627907,0,7,10,0.144737,0.040609


In [5]:
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    required = ['id', 'title', 'label']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Missing required feature columns: {missing}')

    if df['label'].dtype == object:
        label_map = {
            'clickbait': 1,
            'non-clickbait': 0,
            'non clickbait': 0,
            '1': 1,
            '0': 0,
        }
        df['label'] = df['label'].astype(str).str.strip().str.lower().map(label_map)

    if df['label'].isna().any():
        bad = df.loc[df['label'].isna(), ['id', 'label']].head()
        raise ValueError(f'Invalid labels detected:\n{bad}')

    df['label'] = df['label'].astype(int)
    df['label_str'] = df['label'].map({1: 'Clickbait', 0: 'Non-clickbait'})
    df['title'] = df['title'].fillna('').astype(str)
    if 'lead_paragraph' not in df.columns:
        df['lead_paragraph'] = ''
    df['lead_paragraph'] = df['lead_paragraph'].fillna('').astype(str)
    df['text_title'] = df['title']
    df['text_title_lead'] = (df['title'] + ' ' + df['lead_paragraph']).str.strip()

    if 'title_char_len' not in df.columns:
        df['title_char_len'] = df['title'].str.len()
    if 'title_n_cb_keywords' not in df.columns:
        df['title_n_cb_keywords'] = 0
    if 'title_n_curiosity_gap_cues' not in df.columns:
        df['title_n_curiosity_gap_cues'] = 0
    if 'title_n_question' not in df.columns:
        df['title_n_question'] = df['title'].str.count(r'\?')

    return df

features_df = prepare_features(features_df)

if features_df['id'].duplicated().any():
    duplicated = features_df.loc[features_df['id'].duplicated(keep=False), 'id'].unique()[:10]
    raise ValueError(f'Duplicate ids in feature dataframe: {duplicated}')

print(features_df[['id', 'title', 'label', 'label_str', 'source', 'category']].head())
display(features_df['label_str'].value_counts().to_frame('count'))

             id                                              title  label  \
0  article_0001          Sân bay Vinh đóng cửa 6 tháng để nâng cấp      0   
1  article_0002               5 người thoát nạn khi ôtô bị lũ cuốn      0   
2  article_0004         Quy hoạch 9 khu chức năng phía đông TP HCM      0   
3  article_0006  Tổng Bí thư trao quyết định thành lập Cơ quan ...      0   
4  article_0010  Quốc hội bước vào tuần làm việc cuối cùng của ...      0   

       label_str     source          category  
0  Non-clickbait  VnExpress  Tin tức tổng hợp  
1  Non-clickbait  VnExpress  Tin tức tổng hợp  
2  Non-clickbait  VnExpress  Tin tức tổng hợp  
3  Non-clickbait  VnExpress  Tin tức tổng hợp  
4  Non-clickbait  VnExpress  Tin tức tổng hợp  


,count
label_str,
Non-clickbait,2349
Clickbait,1065


## Utily Function

In [6]:
def attach_features(split_df: pd.DataFrame, features_df: pd.DataFrame) -> pd.DataFrame:
    split_cols = [col for col in split_df.columns if col != 'label_str']
    merged = split_df[split_cols].merge(
        features_df,
        on='id',
        how='left',
        suffixes=('_split', ''),
    )

    if merged['title'].isna().any():
        missing_ids = merged.loc[merged['title'].isna(), 'id'].head().tolist()
        raise ValueError(f'Split contains ids not found in features_df. Examples: {missing_ids}')

    if 'label_split' in merged.columns:
        mismatch = merged['label_split'].astype(int) != merged['label'].astype(int)
        if mismatch.any():
            bad = merged.loc[mismatch, ['id', 'label_split', 'label']].head()
            raise ValueError(f'Label mismatch between split and features:\n{bad}')

    return merged


def get_random_split_data(split_df: pd.DataFrame, features_df: pd.DataFrame):
    merged = attach_features(split_df, features_df)
    train_df = merged[merged['split'] == 'train'].copy()
    val_df = merged[merged['split'] == 'validation'].copy()
    test_df = merged[merged['split'] == 'test'].copy()
    return train_df, val_df, test_df


train_df, val_df, test_df = get_random_split_data(random_split_df, features_df)
print('Random split sizes:')
print('train:', train_df.shape, 'validation:', val_df.shape, 'test:', test_df.shape)
display(pd.concat([
    train_df.assign(part='train'),
    val_df.assign(part='validation'),
    test_df.assign(part='test'),
]).groupby(['part', 'label_str']).size().unstack(fill_value=0))

Random split sizes:
train: (2389, 46) validation: (342, 46) test: (683, 46)


label_str,Clickbait,Non-clickbait
part,,
test,213,470
train,745,1644
validation,107,235


In [7]:
CONFUSION_MATRICES: dict[str, Any] = {}


def compute_metrics(y_true, y_pred, y_score=None) -> dict[str, Any]:
    precision, recall, f1_clickbait, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        pos_label=1,
        average='binary',
        zero_division=0,
    )

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'clickbait_precision': precision,
        'clickbait_recall': recall,
        'clickbait_f1': f1_clickbait,
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_score)
        except Exception:
            metrics['roc_auc'] = np.nan
        try:
            metrics['pr_auc'] = average_precision_score(y_true, y_score)
        except Exception:
            metrics['pr_auc'] = np.nan
    else:
        metrics['roc_auc'] = np.nan
        metrics['pr_auc'] = np.nan

    return metrics


def confusion_to_dict(y_true, y_pred) -> dict[str, int]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}


def get_model_score(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X)
    return None


def make_result_row(
    *,
    protocol: str,
    run_name: str,
    model_name: str,
    feature_set: str,
    train_size: int,
    eval_size: int,
    y_true,
    y_pred,
    y_score=None,
) -> dict[str, Any]:
    row = {
        'protocol': protocol,
        'run_name': run_name,
        'model_name': model_name,
        'feature_set': feature_set,
        'train_size': train_size,
        'eval_size': eval_size,
    }
    row.update(compute_metrics(y_true, y_pred, y_score))
    return row


def add_confusion_matrix(protocol: str, run_name: str, model_name: str, y_true, y_pred):
    key = f'{protocol}::{run_name}::{model_name}'
    CONFUSION_MATRICES[key] = confusion_to_dict(y_true, y_pred)

## BASELINE

In [8]:
CB_KEYWORDS = [
    'sốc', 'kinh hoàng', 'khó tin', 'bất ngờ', 'không ngờ', 'choáng',
    'bí mật', 'tiết lộ', 'hé lộ', 'sự thật', 'vì sao', 'tại sao',
    'lý do', 'gây sốt', 'gây bão', 'gây tranh cãi', 'hot', 'nóng',
    'cảnh báo', 'nguy hiểm', 'đáng lo', 'bí quyết', 'mẹo', 'điều bạn',
]


def keyword_score(df: pd.DataFrame) -> np.ndarray:
    if {'title_n_cb_keywords', 'title_n_curiosity_gap_cues', 'title_n_question'}.issubset(df.columns):
        return (
            df['title_n_cb_keywords'].fillna(0).to_numpy()
            + df['title_n_curiosity_gap_cues'].fillna(0).to_numpy()
            + df['title_n_question'].fillna(0).to_numpy()
        )

    text = df['title'].fillna('').str.lower()
    scores = np.zeros(len(df), dtype=float)
    for keyword in CB_KEYWORDS:
        scores += text.str.contains(keyword, regex=False).astype(int).to_numpy()
    scores += text.str.contains(r'\?', regex=True).astype(int).to_numpy()
    return scores


def fit_threshold(train_scores, train_y, val_scores, val_y, metric='macro_f1') -> float:
    unique_scores = np.unique(np.concatenate([train_scores, val_scores]))
    candidate_thresholds = sorted(set(unique_scores.tolist() + [0, 1]))

    best_threshold = candidate_thresholds[0]
    best_score = -1
    for threshold in candidate_thresholds:
        pred = (val_scores >= threshold).astype(int)
        if metric == 'clickbait_f1':
            score = f1_score(val_y, pred, pos_label=1, zero_division=0)
        else:
            score = f1_score(val_y, pred, average='macro', zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = threshold
    return float(best_threshold)


def evaluate_simple_baselines_random(train_df, val_df, test_df) -> tuple[pd.DataFrame, pd.DataFrame]:
    results = []
    prediction_frames = []
    rng = np.random.default_rng(RANDOM_STATE)

    y_train = train_df['label'].to_numpy()
    class_counts = pd.Series(y_train).value_counts(normalize=True).sort_index()
    majority_label = int(pd.Series(y_train).mode().iloc[0])
    class_probs = np.array([class_counts.get(0, 0.0), class_counts.get(1, 0.0)])

    length_threshold = fit_threshold(
        train_df['title_char_len'].to_numpy(), y_train,
        val_df['title_char_len'].to_numpy(), val_df['label'].to_numpy(),
    )
    keyword_threshold = fit_threshold(
        keyword_score(train_df), y_train,
        keyword_score(val_df), val_df['label'].to_numpy(),
    )

    for eval_name, eval_df in [('validation', val_df), ('test', test_df)]:
        y_true = eval_df['label'].to_numpy()
        outputs = []
        outputs.append(('majority_class', np.full(len(eval_df), majority_label), None))
        outputs.append(('random_stratified', rng.choice([0, 1], size=len(eval_df), p=class_probs), None))

        length_scores = eval_df['title_char_len'].to_numpy()
        outputs.append((
            f'length_heuristic_threshold_{length_threshold:.2f}',
            (length_scores >= length_threshold).astype(int),
            length_scores,
        ))

        kw_scores = keyword_score(eval_df)
        outputs.append((
            f'keyword_heuristic_threshold_{keyword_threshold:.2f}',
            (kw_scores >= keyword_threshold).astype(int),
            kw_scores,
        ))

        for model_name, y_pred, y_score in outputs:
            results.append(make_result_row(
                protocol='random_split', run_name=eval_name, model_name=model_name,
                feature_set='simple_rule', train_size=len(train_df), eval_size=len(eval_df),
                y_true=y_true, y_pred=y_pred, y_score=y_score,
            ))
            add_confusion_matrix('random_split', eval_name, model_name, y_true, y_pred)
            prediction_frames.append(pd.DataFrame({
                'id': eval_df['id'].values,
                'protocol': 'random_split',
                'run_name': eval_name,
                'model_name': model_name,
                'y_true': y_true,
                'y_pred': y_pred,
                'y_score': y_score if y_score is not None else np.nan,
            }))

    return pd.DataFrame(results), pd.concat(prediction_frames, ignore_index=True)


simple_random_results, simple_random_predictions = evaluate_simple_baselines_random(train_df, val_df, test_df)
display(simple_random_results.sort_values(['run_name', 'macro_f1'], ascending=[True, False]))

,protocol,run_name,model_name,feature_set,train_size,eval_size,accuracy,balanced_accuracy,macro_f1,weighted_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc
7,random_split,test,keyword_heuristic_threshold_1.00,simple_rule,2389,683,0.755490,0.658041,0.671079,0.733778,0.685484,0.399061,0.504451,0.661143,0.476681
5,random_split,test,random_stratified,simple_rule,2389,683,0.575403,0.507936,0.507876,0.576470,0.322581,0.328638,0.325581,NaN,NaN
6,random_split,test,length_heuristic_threshold_73.00,simple_rule,2389,683,0.556369,0.497957,0.497371,0.562168,0.309322,0.342723,0.325167,0.478898,0.297721
4,random_split,test,majority_class,simple_rule,2389,683,0.688141,0.500000,0.407632,0.561017,0.000000,0.000000,0.000000,NaN,NaN
3,random_split,validation,keyword_heuristic_threshold_1.00,simple_rule,2389,342,0.751462,0.648618,0.660536,0.726290,0.689655,0.373832,0.484848,0.649234,0.463144
2,random_split,validation,length_heuristic_threshold_73.00,simple_rule,2389,342,0.576023,0.528594,0.526103,0.583669,0.346774,0.401869,0.372294,0.531100,0.336877
1,random_split,validation,random_stratified,simple_rule,2389,342,0.584795,0.512070,0.512116,0.582593,0.330097,0.317757,0.323810,NaN,NaN
0,random_split,validation,majority_class,simple_rule,2389,342,0.687135,0.500000,0.407279,0.559711,0.000000,0.000000,0.000000,NaN,NaN


## TF-IDF Vectorizers and MODEL builders

In [9]:
def build_vectorizer(vectorizer_type: str):
    if vectorizer_type == 'word':
        return TfidfVectorizer(
            analyzer='word', ngram_range=(1, 2), min_df=2, max_df=0.95,
            sublinear_tf=True, lowercase=True,
        )
    if vectorizer_type == 'char':
        return TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_df=0.95,
            sublinear_tf=True, lowercase=True,
        )
    if vectorizer_type == 'word_char':
        return FeatureUnion([
            ('word', build_vectorizer('word')),
            ('char', build_vectorizer('char')),
        ])
    raise ValueError(f'Unknown vectorizer_type: {vectorizer_type}')


def build_classifier(classifier_type: str):
    if classifier_type == 'logreg':
        return LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    if classifier_type == 'svm':
        return LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)
    if classifier_type == 'nb':
        return MultinomialNB()
    if classifier_type == 'rf':
        return RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    raise ValueError(f'Unknown classifier_type: {classifier_type}')


def build_pipeline(vectorizer_type: str, classifier_type: str) -> Pipeline:
    return Pipeline([
        ('vectorizer', build_vectorizer(vectorizer_type)),
        ('classifier', build_classifier(classifier_type)),
    ])


MODEL_SPECS = [
    {'model_name': 'tfidf_word_logreg', 'vectorizer': 'word', 'classifier': 'logreg'},
    {'model_name': 'tfidf_char_logreg', 'vectorizer': 'char', 'classifier': 'logreg'},
    {'model_name': 'tfidf_word_char_logreg', 'vectorizer': 'word_char', 'classifier': 'logreg'},
    {'model_name': 'tfidf_word_svm', 'vectorizer': 'word', 'classifier': 'svm'},
    {'model_name': 'tfidf_char_svm', 'vectorizer': 'char', 'classifier': 'svm'},
    {'model_name': 'tfidf_word_char_svm', 'vectorizer': 'word_char', 'classifier': 'svm'},
    {'model_name': 'tfidf_word_nb', 'vectorizer': 'word', 'classifier': 'nb'},
    {'model_name': 'tfidf_word_rf', 'vectorizer': 'word', 'classifier': 'rf'},
]

KFOLD_MODEL_SPECS = [spec for spec in MODEL_SPECS if spec['classifier'] != 'rf']
MODEL_SPECS

[{'model_name': 'tfidf_word_logreg',
  'vectorizer': 'word',
  'classifier': 'logreg'},
 {'model_name': 'tfidf_char_logreg',
  'vectorizer': 'char',
  'classifier': 'logreg'},
 {'model_name': 'tfidf_word_char_logreg',
  'vectorizer': 'word_char',
  'classifier': 'logreg'},
 {'model_name': 'tfidf_word_svm', 'vectorizer': 'word', 'classifier': 'svm'},
 {'model_name': 'tfidf_char_svm', 'vectorizer': 'char', 'classifier': 'svm'},
 {'model_name': 'tfidf_word_char_svm',
  'vectorizer': 'word_char',
  'classifier': 'svm'},
 {'model_name': 'tfidf_word_nb', 'vectorizer': 'word', 'classifier': 'nb'},
 {'model_name': 'tfidf_word_rf', 'vectorizer': 'word', 'classifier': 'rf'}]

## Evaluation Helpers for SKlearn Models

In [10]:
def evaluate_sklearn_model(
    *, model: Pipeline, train_df: pd.DataFrame, eval_df: pd.DataFrame,
    model_name: str, protocol: str, run_name: str,
    text_col: str = 'text_title', collect_predictions: bool = False,
):
    X_train = train_df[text_col].fillna('').astype(str)
    y_train = train_df['label'].astype(int).to_numpy()
    X_eval = eval_df[text_col].fillna('').astype(str)
    y_true = eval_df['label'].astype(int).to_numpy()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_eval)
    y_score = get_model_score(model, X_eval)

    row = make_result_row(
        protocol=protocol, run_name=run_name, model_name=model_name,
        feature_set=text_col, train_size=len(train_df), eval_size=len(eval_df),
        y_true=y_true, y_pred=y_pred, y_score=y_score,
    )
    add_confusion_matrix(protocol, run_name, model_name, y_true, y_pred)

    pred_df = None
    if collect_predictions:
        pred_df = pd.DataFrame({
            'id': eval_df['id'].values,
            'protocol': protocol,
            'run_name': run_name,
            'model_name': model_name,
            'y_true': y_true,
            'y_pred': y_pred,
            'y_score': y_score if y_score is not None else np.nan,
        })

    return row, pred_df, model

## RANDOM SPLIT BENCHMARK

In [11]:
random_results = [simple_random_results]
random_prediction_frames = [simple_random_predictions]
fitted_random_models = {}

for spec in MODEL_SPECS:
    model = build_pipeline(spec['vectorizer'], spec['classifier'])
    for eval_name, eval_df in [('validation', val_df), ('test', test_df)]:
        row, pred_df, fitted_model = evaluate_sklearn_model(
            model=clone(model), train_df=train_df, eval_df=eval_df,
            model_name=spec['model_name'], protocol='random_split', run_name=eval_name,
            text_col='text_title', collect_predictions=True,
        )
        random_results.append(pd.DataFrame([row]))
        random_prediction_frames.append(pred_df)
        if eval_name == 'test':
            fitted_random_models[spec['model_name']] = fitted_model

random_split_results = pd.concat(random_results, ignore_index=True)
predictions_random_split = pd.concat(random_prediction_frames, ignore_index=True)
random_split_results = random_split_results.sort_values(['run_name', 'macro_f1'], ascending=[True, False]).reset_index(drop=True)
display(random_split_results)

,protocol,run_name,model_name,feature_set,train_size,eval_size,accuracy,balanced_accuracy,macro_f1,weighted_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc
0,random_split,test,tfidf_word_logreg,text_title,2389,683,0.751098,0.742134,0.725923,0.757179,0.581749,0.718310,0.642857,0.814954,0.656721
1,random_split,test,tfidf_word_char_logreg,text_title,2389,683,0.749634,0.730801,0.720058,0.754297,0.584677,0.680751,0.629067,0.819079,0.666477
2,random_split,test,tfidf_word_svm,text_title,2389,683,0.754026,0.722440,0.718378,0.756080,0.599119,0.638498,0.618182,0.802467,0.639669
3,random_split,test,tfidf_word_char_svm,text_title,2389,683,0.756955,0.718150,0.717589,0.757263,0.609302,0.615023,0.612150,0.803866,0.649234
4,random_split,test,tfidf_char_svm,text_title,2389,683,0.749634,0.723100,0.716338,0.752907,0.588983,0.652582,0.619154,0.801878,0.649820
5,random_split,test,tfidf_char_logreg,text_title,2389,683,0.740849,0.726985,0.713219,0.746714,0.569767,0.690141,0.624204,0.811707,0.656415
6,random_split,test,tfidf_word_rf,text_title,2389,683,0.755490,0.676011,0.687816,0.742509,0.651316,0.464789,0.542466,0.812701,0.654530
7,random_split,test,keyword_heuristic_threshold_1.00,simple_rule,2389,683,0.755490,0.658041,0.671079,0.733778,0.685484,0.399061,0.504451,0.661143,0.476681
8,random_split,test,tfidf_word_nb,text_title,2389,683,0.742313,0.607392,0.606762,0.693637,0.768116,0.248826,0.375887,0.806313,0.641150
9,random_split,test,random_stratified,simple_rule,2389,683,0.575403,0.507936,0.507876,0.576470,0.322581,0.328638,0.325581,NaN,NaN


In [12]:
test_model_results = random_split_results[
    (random_split_results['run_name'] == 'test')
    & (random_split_results['model_name'].str.startswith('tfidf_'))
].copy()


def select_best_spec_by_classifier(test_results: pd.DataFrame, classifier: str):
    candidate_names = [spec['model_name'] for spec in MODEL_SPECS if spec['classifier'] == classifier]
    rows = test_results[test_results['model_name'].isin(candidate_names)].copy()
    if rows.empty:
        raise ValueError(f'No test result found for classifier={classifier}')
    best_row = rows.sort_values('macro_f1', ascending=False).iloc[0]
    best_spec = next(spec for spec in MODEL_SPECS if spec['model_name'] == best_row['model_name'])
    return best_row, best_spec


best_traditional_row = test_model_results.sort_values('macro_f1', ascending=False).iloc[0]
BEST_MODEL_NAME = best_traditional_row['model_name']
BEST_MODEL_SPEC = next(spec for spec in MODEL_SPECS if spec['model_name'] == BEST_MODEL_NAME)

BEST_LOGREG_ROW, BEST_LOGREG_SPEC = select_best_spec_by_classifier(test_model_results, 'logreg')
BEST_SVM_ROW, BEST_SVM_SPEC = select_best_spec_by_classifier(test_model_results, 'svm')
DOMAIN_MODEL_SPECS = [BEST_LOGREG_SPEC, BEST_SVM_SPEC]

print('Best traditional model on random test split:')
display(best_traditional_row.to_frame().T)

print('Models used for domain generalization:')
display(pd.DataFrame([
    {
        'role': 'best_logreg',
        'model_name': BEST_LOGREG_SPEC['model_name'],
        'vectorizer': BEST_LOGREG_SPEC['vectorizer'],
        'random_test_macro_f1': BEST_LOGREG_ROW['macro_f1'],
        'random_test_clickbait_f1': BEST_LOGREG_ROW['clickbait_f1'],
    },
    {
        'role': 'best_svm',
        'model_name': BEST_SVM_SPEC['model_name'],
        'vectorizer': BEST_SVM_SPEC['vectorizer'],
        'random_test_macro_f1': BEST_SVM_ROW['macro_f1'],
        'random_test_clickbait_f1': BEST_SVM_ROW['clickbait_f1'],
    },
]))

best_traditional_predictions = predictions_random_split[
    (predictions_random_split['run_name'] == 'test')
    & (predictions_random_split['model_name'] == BEST_MODEL_NAME)
].copy()

Best traditional model on random test split:


,protocol,run_name,model_name,feature_set,train_size,eval_size,accuracy,balanced_accuracy,macro_f1,weighted_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc
0,random_split,test,tfidf_word_logreg,text_title,2389,683,0.751098,0.742134,0.725923,0.757179,0.581749,0.71831,0.642857,0.814954,0.656721


Models used for domain generalization:


,role,model_name,vectorizer,random_test_macro_f1,random_test_clickbait_f1
0,best_logreg,tfidf_word_logreg,word,0.725923,0.642857
1,best_svm,tfidf_word_svm,word,0.718378,0.618182


## Stratified 5-fold benchmark

In [13]:
def run_kfold_benchmark(features_df: pd.DataFrame, kfold_df: pd.DataFrame) -> pd.DataFrame:
    merged = attach_features(kfold_df, features_df)
    rows = []

    for fold in sorted(merged['fold'].unique()):
        train_fold = merged[merged['fold'] != fold].copy()
        eval_fold = merged[merged['fold'] == fold].copy()
        y_true = eval_fold['label'].to_numpy()

        majority_label = int(train_fold['label'].mode().iloc[0])
        y_pred = np.full(len(eval_fold), majority_label)
        rows.append(make_result_row(
            protocol='stratified_kfold', run_name=f'fold_{fold}', model_name='majority_class',
            feature_set='simple_rule', train_size=len(train_fold), eval_size=len(eval_fold),
            y_true=y_true, y_pred=y_pred,
        ))
        add_confusion_matrix('stratified_kfold', f'fold_{fold}', 'majority_class', y_true, y_pred)

        for spec in KFOLD_MODEL_SPECS:
            row, _, _ = evaluate_sklearn_model(
                model=build_pipeline(spec['vectorizer'], spec['classifier']),
                train_df=train_fold, eval_df=eval_fold, model_name=spec['model_name'],
                protocol='stratified_kfold', run_name=f'fold_{fold}', text_col='text_title',
                collect_predictions=False,
            )
            rows.append(row)

    return pd.DataFrame(rows)


kfold_results = run_kfold_benchmark(features_df, kfold_df)
display(kfold_results.head())
display(kfold_results.groupby('model_name')[['macro_f1', 'clickbait_f1', 'balanced_accuracy', 'roc_auc', 'pr_auc']].agg(['mean', 'std']).sort_values(('macro_f1', 'mean'), ascending=False))

,protocol,run_name,model_name,feature_set,train_size,eval_size,accuracy,balanced_accuracy,macro_f1,weighted_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc
0,stratified_kfold,fold_0,majority_class,simple_rule,2731,683,0.688141,0.500000,0.407632,0.561017,0.000000,0.000000,0.000000,NaN,NaN
1,stratified_kfold,fold_0,tfidf_word_logreg,text_title,2731,683,0.775988,0.760219,0.748989,0.779966,0.621951,0.718310,0.666667,0.848836,0.697137
2,stratified_kfold,fold_0,tfidf_char_logreg,text_title,2731,683,0.786237,0.767666,0.758627,0.789345,0.640167,0.718310,0.676991,0.838298,0.704312
3,stratified_kfold,fold_0,tfidf_word_char_logreg,text_title,2731,683,0.773060,0.746539,0.741091,0.775324,0.626087,0.676056,0.650113,0.848347,0.707967
4,stratified_kfold,fold_0,tfidf_word_svm,text_title,2731,683,0.765739,0.718115,0.721968,0.763478,0.633166,0.591549,0.611650,0.834482,0.682533


macro_f1           clickbait_f1            \
                            mean       std         mean       std   
model_name                                                          
tfidf_word_char_logreg  0.746438  0.004844     0.660068  0.007559   
tfidf_word_logreg       0.745244  0.010330     0.662561  0.014120   
tfidf_word_char_svm     0.742187  0.022359     0.646654  0.028602   
tfidf_char_logreg       0.740359  0.012838     0.656144  0.013980   
tfidf_word_svm          0.740119  0.017125     0.644574  0.026859   
tfidf_char_svm          0.735856  0.021299     0.640047  0.028226   
tfidf_word_nb           0.623440  0.026892     0.404264  0.044668   
majority_class          0.407600  0.000072     0.000000  0.000000   

                       balanced_accuracy             roc_auc            \
                                    mean       std      mean       std   
model_name                                                               
tfidf_word_char_logreg          0.754529  0.006220  0.841945  0.013161   
tfidf_word_logreg               0.757153  0.011312  0.837537  0.012647   
tfidf_word_char_svm             0.743224  0.021401  0.827254  0.012980   
tfidf_char_logreg               0.751902  0.010644  0.833872  0.014327   
tfidf_word_svm                  0.742296  0.020128  0.825003  0.012831   
tfidf_char_svm                  0.738631  0.021234  0.825022  0.015366   
tfidf_word_nb                   0.619928  0.020293  0.827472  0.018017   
majority_class                  0.500000  0.000000       NaN       NaN   

                          pr_auc            
                            mean       std  
model_name                                  
tfidf_word_char_logreg  0.711141  0.039062  
tfidf_word_logreg       0.702516  0.035408  
tfidf_word_char_svm     0.698399  0.040614  
tfidf_char_logreg       0.700857  0.038981  
tfidf_word_svm          0.689792  0.042344  
tfidf_char_svm          0.695286  0.040985  
tfidf_word_nb           0.689704  0.037661  
majority_class               NaN       NaN

## 9. Leave-one-source-out benchmark

In [14]:
def run_source_heldout(features_df: pd.DataFrame, heldout_df: pd.DataFrame, specs: list[dict]) -> pd.DataFrame:
    merged = attach_features(heldout_df, features_df)
    rows = []
    for spec in specs:
        for source in sorted(merged['heldout_source'].unique()):
            run_df = merged[merged['heldout_source'] == source].copy()
            train_run = run_df[run_df['split'] == 'train'].copy()
            test_run = run_df[run_df['split'] == 'test'].copy()
            row, _, _ = evaluate_sklearn_model(
                model=build_pipeline(spec['vectorizer'], spec['classifier']),
                train_df=train_run, eval_df=test_run, model_name=spec['model_name'],
                protocol='leave_one_source_out', run_name=source, text_col='text_title',
                collect_predictions=False,
            )
            row['heldout_source'] = source
            row['domain_model_role'] = f"best_{spec['classifier']}"
            rows.append(row)
    return pd.DataFrame(rows)


source_heldout_results = run_source_heldout(features_df, source_heldout_df, DOMAIN_MODEL_SPECS)
display(source_heldout_results.sort_values(['macro_f1', 'model_name']))

display(
    source_heldout_results
    .groupby('model_name')[['macro_f1', 'clickbait_f1', 'balanced_accuracy']]
    .agg(['mean', 'std', 'min'])
    .sort_values(('macro_f1', 'mean'), ascending=False)
)

,protocol,run_name,model_name,feature_set,train_size,eval_size,accuracy,balanced_accuracy,macro_f1,weighted_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc,heldout_source,domain_model_role
7,leave_one_source_out,VnExpress,tfidf_word_logreg,text_title,2806,608,0.720395,0.742133,0.628816,0.759816,0.311927,0.772727,0.444444,0.798885,0.364191,VnExpress,best_logreg
15,leave_one_source_out,VnExpress,tfidf_word_svm,text_title,2806,608,0.746711,0.729196,0.640932,0.779406,0.326316,0.704545,0.446043,0.778781,0.318057,VnExpress,best_svm
11,leave_one_source_out,SaoStar,tfidf_word_svm,text_title,3128,286,0.674825,0.703763,0.670438,0.681872,0.849624,0.607527,0.708464,0.781774,0.870522,SaoStar,best_svm
14,leave_one_source_out,VietNamNet,tfidf_word_svm,text_title,3058,356,0.721910,0.689774,0.685588,0.724612,0.557377,0.601770,0.578723,0.773626,0.668417,VietNamNet,best_svm
12,leave_one_source_out,Thanh Niên,tfidf_word_svm,text_title,2838,576,0.737847,0.691256,0.689645,0.738909,0.559322,0.575581,0.567335,0.791547,0.593816,Thanh Niên,best_svm
4,leave_one_source_out,Thanh Niên,tfidf_word_logreg,text_title,2838,576,0.741319,0.705417,0.699444,0.744631,0.560847,0.616279,0.587258,0.800599,0.601033,Thanh Niên,best_logreg
6,leave_one_source_out,VietNamNet,tfidf_word_logreg,text_title,3058,356,0.727528,0.712826,0.700469,0.733344,0.558824,0.672566,0.610442,0.778470,0.677312,VietNamNet,best_logreg
3,leave_one_source_out,SaoStar,tfidf_word_logreg,text_title,3128,286,0.723776,0.729839,0.712351,0.729589,0.840764,0.709677,0.769679,0.800538,0.879990,SaoStar,best_logreg
8,leave_one_source_out,24h,tfidf_word_svm,text_title,2850,564,0.718085,0.714577,0.714290,0.718260,0.677291,0.685484,0.681363,0.787477,0.736274,24h,best_svm
0,leave_one_source_out,24h,tfidf_word_logreg,text_title,2850,564,0.725177,0.729150,0.724345,0.726171,0.663158,0.762097,0.709193,0.803466,0.753650,24h,best_logreg


macro_f1                     clickbait_f1            \
                       mean       std       min         mean       std   
model_name                                                               
tfidf_word_logreg  0.713953  0.040971  0.628816     0.652272  0.132122   
tfidf_word_svm     0.707558  0.043342  0.640932     0.635124  0.123326   

                            balanced_accuracy                      
                        min              mean       std       min  
model_name                                                         
tfidf_word_logreg  0.444444          0.736248  0.030558  0.705417  
tfidf_word_svm     0.446043          0.728490  0.041708  0.689774

## Category-held-out benchmark

In [15]:
def run_category_heldout(features_df: pd.DataFrame, heldout_df: pd.DataFrame, specs: list[dict]) -> pd.DataFrame:
    merged = attach_features(heldout_df, features_df)
    rows = []
    for spec in specs:
        for category in sorted(merged['heldout_category'].unique()):
            run_df = merged[merged['heldout_category'] == category].copy()
            train_run = run_df[run_df['split'] == 'train'].copy()
            test_run = run_df[run_df['split'] == 'test'].copy()
            row, _, _ = evaluate_sklearn_model(
                model=build_pipeline(spec['vectorizer'], spec['classifier']),
                train_df=train_run, eval_df=test_run, model_name=spec['model_name'],
                protocol='category_heldout', run_name=category, text_col='text_title',
                collect_predictions=False,
            )
            row['heldout_category'] = category
            row['domain_model_role'] = f"best_{spec['classifier']}"
            rows.append(row)
    return pd.DataFrame(rows)


category_heldout_results = run_category_heldout(features_df, category_heldout_df, DOMAIN_MODEL_SPECS)
display(category_heldout_results.sort_values(['macro_f1', 'model_name']))

display(
    category_heldout_results
    .groupby('model_name')[['macro_f1', 'clickbait_f1', 'balanced_accuracy']]
    .agg(['mean', 'std', 'min'])
    .sort_values(('macro_f1', 'mean'), ascending=False)
)

,protocol,run_name,model_name,feature_set,train_size,eval_size,accuracy,balanced_accuracy,macro_f1,weighted_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc,heldout_category,domain_model_role
16,category_heldout,Giải trí & Showbiz,tfidf_word_svm,text_title,3176,238,0.588235,0.608333,0.570777,0.606421,0.796610,0.559524,0.657343,0.703401,0.842508,Giải trí & Showbiz,best_svm
19,category_heldout,Quốc tế,tfidf_word_svm,text_title,3173,241,0.643154,0.612017,0.586802,0.663415,0.358696,0.550000,0.434211,0.641897,0.415671,Quốc tế,best_svm
4,category_heldout,Giải trí & Showbiz,tfidf_word_logreg,text_title,3176,238,0.626050,0.610119,0.591735,0.640473,0.784173,0.648810,0.710098,0.689881,0.827147,Giải trí & Showbiz,best_logreg
7,category_heldout,Quốc tế,tfidf_word_logreg,text_title,3173,241,0.668050,0.617449,0.600663,0.683025,0.378049,0.516667,0.436620,0.690516,0.476122,Quốc tế,best_logreg
10,category_heldout,Tin tức tổng hợp,tfidf_word_logreg,text_title,3217,197,0.837563,0.855489,0.606197,0.883530,0.184211,0.875000,0.304348,0.882275,0.354575,Tin tức tổng hợp,best_logreg
22,category_heldout,Tin tức tổng hợp,tfidf_word_svm,text_title,3217,197,0.842640,0.858135,0.611143,0.886807,0.189189,0.875000,0.311111,0.904762,0.451846,Tin tức tổng hợp,best_svm
9,category_heldout,Thể thao,tfidf_word_logreg,text_title,3089,325,0.686154,0.629396,0.618334,0.696055,0.413462,0.511905,0.457447,0.692946,0.508034,Thể thao,best_logreg
21,category_heldout,Thể thao,tfidf_word_svm,text_title,3089,325,0.692308,0.660690,0.638889,0.705983,0.431034,0.595238,0.500000,0.711421,0.520396,Thể thao,best_svm
12,category_heldout,Chính trị & An ninh,tfidf_word_svm,text_title,3241,173,0.809249,0.651341,0.653433,0.807905,0.428571,0.413793,0.421053,0.843630,0.559754,Chính trị & An ninh,best_svm
13,category_heldout,Công nghệ & Khoa học,tfidf_word_svm,text_title,3194,220,0.686364,0.679036,0.669993,0.691375,0.548387,0.653846,0.596491,0.772842,0.629167,Công nghệ & Khoa học,best_svm


macro_f1                     clickbait_f1            \
                       mean       std       min         mean       std   
model_name                                                               
tfidf_word_logreg  0.687978  0.070603  0.591735     0.597264  0.139455   
tfidf_word_svm     0.679901  0.074971  0.570777     0.577762  0.137610   

                            balanced_accuracy                      
                        min              mean       std       min  
model_name                                                         
tfidf_word_logreg  0.304348          0.718082  0.072500  0.610119  
tfidf_word_svm     0.311111          0.708931  0.076154  0.608333

## LOGISTIC REGRESSION

In [16]:
def extract_feature_names(vectorizer):
    if hasattr(vectorizer, 'get_feature_names_out'):
        return vectorizer.get_feature_names_out()
    if isinstance(vectorizer, FeatureUnion):
        names = []
        for name, transformer in vectorizer.transformer_list:
            if hasattr(transformer, 'get_feature_names_out'):
                names.extend([f'{name}__{feat}' for feat in transformer.get_feature_names_out()])
        return np.array(names)
    raise ValueError('Cannot extract feature names from vectorizer')


def logistic_feature_importance(train_df: pd.DataFrame, spec: dict, top_k: int = 100) -> pd.DataFrame:
    if spec['classifier'] != 'logreg':
        raise ValueError('Feature importance in Phase 4 is defined only for Logistic Regression.')

    model = build_pipeline(spec['vectorizer'], 'logreg')
    X_train = train_df['text_title'].fillna('').astype(str)
    y_train = train_df['label'].astype(int).to_numpy()
    model.fit(X_train, y_train)

    vectorizer = model.named_steps['vectorizer']
    classifier = model.named_steps['classifier']
    feature_names = extract_feature_names(vectorizer)
    coefs = classifier.coef_[0]

    coef_df = pd.DataFrame({
        'model_name': spec['model_name'],
        'vectorizer': spec['vectorizer'],
        'feature': feature_names,
        'coefficient': coefs,
    })
    top_clickbait = coef_df.sort_values('coefficient', ascending=False).head(top_k).assign(direction='clickbait')
    top_non_clickbait = coef_df.sort_values('coefficient', ascending=True).head(top_k).assign(direction='non_clickbait')
    return pd.concat([top_clickbait, top_non_clickbait], ignore_index=True)


feature_importance_logreg = logistic_feature_importance(train_df, BEST_LOGREG_SPEC, top_k=100)
display(feature_importance_logreg.head(20))
display(feature_importance_logreg.tail(20))

,model_name,vectorizer,feature,coefficient,direction
0,tfidf_word_logreg,word,nào,2.101047,clickbait
1,tfidf_word_logreg,word,sao,2.001182,clickbait
2,tfidf_word_logreg,word,có,1.879380,clickbait
3,tfidf_word_logreg,word,bất,1.851198,clickbait
4,tfidf_word_logreg,word,gì,1.837395,clickbait
5,tfidf_word_logreg,word,ngờ,1.710009,clickbait
6,tfidf_word_logreg,word,không,1.680184,clickbait
7,tfidf_word_logreg,word,con,1.625008,clickbait
8,tfidf_word_logreg,word,ăn,1.577031,clickbait
9,tfidf_word_logreg,word,khiến,1.504738,clickbait


,model_name,vectorizer,feature,coefficient,direction
180,tfidf_word_logreg,word,phường,-0.612378,non_clickbait
181,tfidf_word_logreg,word,bắt,-0.608063,non_clickbait
182,tfidf_word_logreg,word,cứu,-0.606216,non_clickbait
183,tfidf_word_logreg,word,ngành,-0.605314,non_clickbait
184,tfidf_word_logreg,word,trợ,-0.604701,non_clickbait
185,tfidf_word_logreg,word,nguy kịch,-0.604167,non_clickbait
186,tfidf_word_logreg,word,vào,-0.601506,non_clickbait
187,tfidf_word_logreg,word,bảng,-0.601000,non_clickbait
188,tfidf_word_logreg,word,cán,-0.600599,non_clickbait
189,tfidf_word_logreg,word,tăng,-0.600590,non_clickbait


In [20]:
# Export kết quả
PHASE4_DIR = Path('phase4')
PHASE4_DIR.mkdir(exist_ok=True)
all_results = pd.concat([
    random_split_results.assign(result_group='random_split'),
    kfold_results.assign(result_group='kfold'),
    source_heldout_results.assign(result_group='source_heldout'),
    category_heldout_results.assign(result_group='category_heldout'),
], ignore_index=True, sort=False)

random_split_results.to_csv(PHASE4_DIR /'random_split_results.csv', index=False)
kfold_results.to_csv(PHASE4_DIR / 'kfold_results.csv', index=False)
source_heldout_results.to_csv(PHASE4_DIR / 'source_heldout_results.csv', index=False)
category_heldout_results.to_csv(PHASE4_DIR / 'category_heldout_results.csv', index=False)
all_results.to_csv(PHASE4_DIR / 'all_results.csv', index=False)

predictions_random_split.to_csv(PHASE4_DIR / 'predictions_random_split.csv', index=False)
best_traditional_predictions.to_csv(PHASE4_DIR / 'best_traditional_model_predictions.csv', index=False)
feature_importance_logreg.to_csv(PHASE4_DIR / 'feature_importance_logreg.csv', index=False)

with open(PHASE4_DIR / 'confusion_matrices.json', 'w', encoding='utf-8') as f:
    json.dump(CONFUSION_MATRICES, f, ensure_ascii=False, indent=2)

print('Exported Phase 4 files:')
for path in sorted(PHASE4_DIR.glob('*')):
    print(f'  - {path}')

Exported Phase 4 files:
  - phase4/all_results.csv
  - phase4/best_traditional_model_predictions.csv
  - phase4/category_heldout_results.csv
  - phase4/confusion_matrices.json
  - phase4/feature_importance_logreg.csv
  - phase4/kfold_results.csv
  - phase4/phase4_summary.md
  - phase4/predictions_random_split.csv
  - phase4/random_split_results.csv
  - phase4/source_heldout_results.csv


In [22]:
# ==================== PHASE4 SUMMARY ====================
def format_float(x):
    if pd.isna(x):
        return 'NA'
    return f'{x:.4f}'

best_random = random_split_results[random_split_results['run_name'] == 'test'].sort_values('macro_f1', ascending=False).iloc[0]
worst_source = source_heldout_results.sort_values('macro_f1').iloc[0]
worst_category = category_heldout_results.sort_values('macro_f1').iloc[0]

domain_model_lines = []
for role, row, spec in [
    ('Best Logistic Regression', BEST_LOGREG_ROW, BEST_LOGREG_SPEC),
    ('Best Linear SVM', BEST_SVM_ROW, BEST_SVM_SPEC),
]:
    domain_model_lines.extend([
        f'- {role}: `{spec["model_name"]}`',
        f'  - Vectorizer: `{spec["vectorizer"]}`',
        f'  - Random-test Macro-F1: {format_float(row["macro_f1"])}',
        f'  - Random-test Clickbait F1: {format_float(row["clickbait_f1"])}',
    ])

summary_lines = [
    '# Phase 4 Benchmark Summary',
    '',
    '## Best Random Split Model',
    '',
    f"- Model: `{best_random['model_name']}`",
    f"- Macro-F1: {format_float(best_random['macro_f1'])}",
    f"- Clickbait F1: {format_float(best_random['clickbait_f1'])}",
    f"- Balanced Accuracy: {format_float(best_random['balanced_accuracy'])}",
    '',
    '## Models Used For Domain Generalization',
    '',
    *domain_model_lines,
    '',
    '## Hardest Source-Held-Out Case',
    '',
    f"- Model: `{worst_source['model_name']}`",
    f"- Held-out source: `{worst_source.get('heldout_source', worst_source['run_name'])}`",
    f"- Macro-F1: {format_float(worst_source['macro_f1'])}",
    f"- Clickbait F1: {format_float(worst_source['clickbait_f1'])}",
    '',
    '## Hardest Category-Held-Out Case',
    '',
    f"- Model: `{worst_category['model_name']}`",
    f"- Held-out category: `{worst_category.get('heldout_category', worst_category['run_name'])}`",
    f"- Macro-F1: {format_float(worst_category['macro_f1'])}",
    f"- Clickbait F1: {format_float(worst_category['clickbait_f1'])}",
    '',
    '## Notes',
    '',
    '- Accuracy is reported but should not be used as the main metric.',
    '- Macro-F1 is the primary metric.',
    '- Source/category metadata is not used as model input in the main benchmark.',
    '- Domain-held-out results should be compared against random split results to quantify robustness drop.',
]

summary_md = '\n'.join(summary_lines) + '\n'
(PHASE4_DIR / 'phase4_summary.md').write_text(summary_md, encoding='utf-8')
print(summary_md)

# Phase 4 Benchmark Summary

## Best Random Split Model

- Model: `tfidf_word_logreg`
- Macro-F1: 0.7259
- Clickbait F1: 0.6429
- Balanced Accuracy: 0.7421

## Models Used For Domain Generalization

- Best Logistic Regression: `tfidf_word_logreg`
  - Vectorizer: `word`
  - Random-test Macro-F1: 0.7259
  - Random-test Clickbait F1: 0.6429
- Best Linear SVM: `tfidf_word_svm`
  - Vectorizer: `word`
  - Random-test Macro-F1: 0.7184
  - Random-test Clickbait F1: 0.6182

## Hardest Source-Held-Out Case

- Model: `tfidf_word_logreg`
- Held-out source: `VnExpress`
- Macro-F1: 0.6288
- Clickbait F1: 0.4444

## Hardest Category-Held-Out Case

- Model: `tfidf_word_svm`
- Held-out category: `Giải trí & Showbiz`
- Macro-F1: 0.5708
- Clickbait F1: 0.6573

## Notes

- Accuracy is reported but should not be used as the main metric.
- Macro-F1 is the primary metric.
- Source/category metadata is not used as model input in the main benchmark.
- Domain-held-out results should be compared against random s

In [23]:
# Sanity Checks
print('Random split test leaderboard:')
display(
    random_split_results[random_split_results['run_name'] == 'test']
    .sort_values('macro_f1', ascending=False)
    [['model_name', 'macro_f1', 'clickbait_f1', 'balanced_accuracy', 'roc_auc', 'pr_auc']]
)

print('K-fold mean/std:')
display(
    kfold_results.groupby('model_name')[['macro_f1', 'clickbait_f1', 'balanced_accuracy']]
    .agg(['mean', 'std'])
    .sort_values(('macro_f1', 'mean'), ascending=False)
)

print('Source-held-out sorted by Macro-F1:')
display(
    source_heldout_results
    .sort_values('macro_f1')
    [['model_name', 'heldout_source', 'macro_f1', 'clickbait_f1', 'balanced_accuracy']]
)

print('Category-held-out sorted by Macro-F1:')
display(
    category_heldout_results
    .sort_values('macro_f1')
    [['model_name', 'heldout_category', 'macro_f1', 'clickbait_f1', 'balanced_accuracy']]
)

Random split test leaderboard:


,model_name,macro_f1,clickbait_f1,balanced_accuracy,roc_auc,pr_auc
0,tfidf_word_logreg,0.725923,0.642857,0.742134,0.814954,0.656721
1,tfidf_word_char_logreg,0.720058,0.629067,0.730801,0.819079,0.666477
2,tfidf_word_svm,0.718378,0.618182,0.722440,0.802467,0.639669
3,tfidf_word_char_svm,0.717589,0.612150,0.718150,0.803866,0.649234
4,tfidf_char_svm,0.716338,0.619154,0.723100,0.801878,0.649820
5,tfidf_char_logreg,0.713219,0.624204,0.726985,0.811707,0.656415
6,tfidf_word_rf,0.687816,0.542466,0.676011,0.812701,0.654530
7,keyword_heuristic_threshold_1.00,0.671079,0.504451,0.658041,0.661143,0.476681
8,tfidf_word_nb,0.606762,0.375887,0.607392,0.806313,0.641150
9,random_stratified,0.507876,0.325581,0.507936,NaN,NaN


K-fold mean/std:


macro_f1           clickbait_f1            \
                            mean       std         mean       std   
model_name                                                          
tfidf_word_char_logreg  0.746438  0.004844     0.660068  0.007559   
tfidf_word_logreg       0.745244  0.010330     0.662561  0.014120   
tfidf_word_char_svm     0.742187  0.022359     0.646654  0.028602   
tfidf_char_logreg       0.740359  0.012838     0.656144  0.013980   
tfidf_word_svm          0.740119  0.017125     0.644574  0.026859   
tfidf_char_svm          0.735856  0.021299     0.640047  0.028226   
tfidf_word_nb           0.623440  0.026892     0.404264  0.044668   
majority_class          0.407600  0.000072     0.000000  0.000000   

                       balanced_accuracy            
                                    mean       std  
model_name                                          
tfidf_word_char_logreg          0.754529  0.006220  
tfidf_word_logreg               0.757153  0.011312  
tfidf_word_char_svm             0.743224  0.021401  
tfidf_char_logreg               0.751902  0.010644  
tfidf_word_svm                  0.742296  0.020128  
tfidf_char_svm                  0.738631  0.021234  
tfidf_word_nb                   0.619928  0.020293  
majority_class                  0.500000  0.000000

Source-held-out sorted by Macro-F1:


,model_name,heldout_source,macro_f1,clickbait_f1,balanced_accuracy
7,tfidf_word_logreg,VnExpress,0.628816,0.444444,0.742133
15,tfidf_word_svm,VnExpress,0.640932,0.446043,0.729196
11,tfidf_word_svm,SaoStar,0.670438,0.708464,0.703763
14,tfidf_word_svm,VietNamNet,0.685588,0.578723,0.689774
12,tfidf_word_svm,Thanh Niên,0.689645,0.567335,0.691256
4,tfidf_word_logreg,Thanh Niên,0.699444,0.587258,0.705417
6,tfidf_word_logreg,VietNamNet,0.700469,0.610442,0.712826
3,tfidf_word_logreg,SaoStar,0.712351,0.769679,0.729839
8,tfidf_word_svm,24h,0.714290,0.681363,0.714577
0,tfidf_word_logreg,24h,0.724345,0.709193,0.729150


Category-held-out sorted by Macro-F1:


,model_name,heldout_category,macro_f1,clickbait_f1,balanced_accuracy
16,tfidf_word_svm,Giải trí & Showbiz,0.570777,0.657343,0.608333
19,tfidf_word_svm,Quốc tế,0.586802,0.434211,0.612017
4,tfidf_word_logreg,Giải trí & Showbiz,0.591735,0.710098,0.610119
7,tfidf_word_logreg,Quốc tế,0.600663,0.436620,0.617449
10,tfidf_word_logreg,Tin tức tổng hợp,0.606197,0.304348,0.855489
22,tfidf_word_svm,Tin tức tổng hợp,0.611143,0.311111,0.858135
9,tfidf_word_logreg,Thể thao,0.618334,0.457447,0.629396
21,tfidf_word_svm,Thể thao,0.638889,0.500000,0.660690
12,tfidf_word_svm,Chính trị & An ninh,0.653433,0.421053,0.651341
13,tfidf_word_svm,Công nghệ & Khoa học,0.669993,0.596491,0.679036
